# 🚀 Full Pipeline: Data Fetching → Training → Backtesting

Complete end-to-end pipeline using modular utilities:
- **Step 1**: Fetch stock prices, SEC fundamentals, and S&P 500 index data (2012-today)
- **Step 2**: Train XGBoost models (2012-2021 training period)
- **Step 3**: Backtest models (2022-today) with 15% stop-loss

Each step includes a summary cell for results analysis.


## 📦 Step 1: Setup and Dependencies

In [1]:
# ============================================================================
# IMPORTS
# ============================================================================

# Core libraries
import pandas as pd
import numpy as np
import warnings
import os
from datetime import datetime

# ============================================================================
# CONFIGURATION - All constants in one place
# ============================================================================

# Data fetching configuration
DATA_START_DATE = '2012-01-03'
# DATA_END_DATE = datetime.now().strftime('%Y-%m-%d')  # Today
DATA_END_DATE = '2025-11-03'


# Training configuration
TRAIN_SPLIT = 0.8  # 80% train, 20% validation
FEATURE_WINDOWS = [5, 10, 15, 20, 25, 30]
TARGET_WINDOWS = [5, 10, 15, 20, 25, 30]
MODEL_TYPES = ['XGBoost']  # Only XGBoost
MODEL_SUFFIX = '2012_2021'  # Tag for models trained on 2012-2021

# Backtesting configuration
BACKTEST_START_DATE = '2022-01-01'
BACKTEST_END_DATE = DATA_END_DATE  # Today
MAX_POSITIONS = 20
STOP_LOSS = 0.15  # 15% stop-loss
TRADE_WHEN_POSITIONS_ZERO = True
RESULTS_DIR = 'data/research/backtesting/2012_2021_models_backtest_01_2022_11_2025'

warnings.filterwarnings('ignore')

# Set up paths
import sys
sys.path.append('.')

# Import our utilities
from fetch_data_util import (
    fetch_stock_prices,
    fetch_sec_fundamentals,
    fetch_sp500_index
)

from training_util import train_all_models_optimized
from backtesting_util import run_ultra_optimized_backtesting


# Docker network configuration (if running in Docker)
import os
if os.path.exists('/.dockerenv'):
    os.environ['PYTHONHTTPSVERIFY'] = '0'
    os.environ['CURL_CA_BUNDLE'] = ''
    print("🐳 Docker environment detected - network settings applied")
    
# Display configuration
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', None)

# ============================================================================
# PRINT CONFIGURATION
# ============================================================================

print("✅ Dependencies loaded successfully!")
print(f"📅 Current time: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print()
print("="*70)
print("📋 CONFIGURATION")
print("="*70)
print(f"📥 Data Fetching: {DATA_START_DATE} to {DATA_END_DATE}")
print(f"🧪 Backtesting: {BACKTEST_START_DATE} to {BACKTEST_END_DATE}")
print(f"🤖 Models: {MODEL_TYPES}")
print(f"🛡️  Stop-Loss: {STOP_LOSS*100:.0f}%")
print(f"📊 Max Positions: {MAX_POSITIONS}")
print(f"📈 Feature Windows: {FEATURE_WINDOWS}")
print(f"🎯 Target Windows: {TARGET_WINDOWS}")
print("="*70)


🐳 Docker environment detected - network settings applied
✅ Dependencies loaded successfully!
📅 Current time: 2025-11-05 07:24:25

📋 CONFIGURATION
📥 Data Fetching: 2012-01-03 to 2025-11-03
🧪 Backtesting: 2022-01-01 to 2025-11-03
🤖 Models: ['XGBoost']
🛡️  Stop-Loss: 15%
📊 Max Positions: 20
📈 Feature Windows: [5, 10, 15, 20, 25, 30]
🎯 Target Windows: [5, 10, 15, 20, 25, 30]


## 📥 Step 2: Fetch All Data (2012 - Today)


In [2]:
print("="*70)
print("📥 STEP 1: DATA FETCHING")
print("="*70)
print(f"📅 Date range: {DATA_START_DATE} to {DATA_END_DATE}")
print(f"📊 Fetching: Stock prices, SEC fundamentals, S&P 500 index")
print()

# 1. Fetch stock prices
print("📈 Fetching stock prices...")
print("-"*70)
stock_data, stock_metadata = fetch_stock_prices(
    start_date=DATA_START_DATE,
    end_date=DATA_END_DATE,
    verbose=True
)

if stock_data is not None:
    print(f"\n✅ Stock prices fetched: {len(stock_data):,} records")
    print(f"   📁 Saved to: {stock_metadata['filepath']}")
    print(f"   📋 Latest copy: {stock_metadata['latest_filepath']}")
else:
    print("\n❌ Failed to fetch stock prices")
    raise Exception("Stock price fetching failed")


📥 STEP 1: DATA FETCHING
📅 Date range: 2012-01-03 to 2025-11-03
📊 Fetching: Stock prices, SEC fundamentals, S&P 500 index

📈 Fetching stock prices...
----------------------------------------------------------------------
📂 Found latest data file, checking if it matches...
   📊 Latest file has range: 2012-01-03 to 2025-11-03 (905,852 records)
   📅 Requested range: 2012-01-03 to 2025-11-03
   ✅ Latest file covers requested range!
   📈 Symbols: 284
   ✅ Using cached data (no re-fetch needed)

✅ Stock prices fetched: 905,852 records
   📁 Saved to: data/research/stock_data_latest/sp500_stock_data_latest.pkl
   📋 Latest copy: data/research/stock_data_latest/sp500_stock_data_latest.pkl


In [5]:
# 2. Fetch SEC fundamentals (requires price data)
print("\n" + "="*70)
print("💰 Fetching SEC fundamentals...")
print("-"*70)

# Get unique tickers from stock data
tickers = stock_data['symbol'].unique().tolist()
print(f"📊 Fetching fundamentals for {len(tickers)} tickers...")

try:
    sec_data, sec_metadata = fetch_sec_fundamentals(
        tickers=tickers,
        price_df=stock_data,
        start_date=DATA_START_DATE,
        end_date=DATA_END_DATE,
        verbose=True,
        resume=True  # Enable resume - continues from checkpoint if interrupted
    )
    
    if sec_data is not None:
        print(f"\n✅ SEC fundamentals fetched: {len(sec_data):,} records")
        print(f"   📁 Saved to: {sec_metadata['filepath']}")
        print(f"   📋 Latest copy: {sec_metadata['latest_filepath']}")
    else:
        print("\n⚠️  SEC fundamentals fetch returned no data")
        print("   Continuing without SEC data...")
        sec_data = None
        sec_metadata = None
except Exception as e:
    print(f"\n⚠️  SEC fundamentals fetch failed: {str(e)}")
    print("   Continuing without SEC data...")
    sec_data = None
    sec_metadata = None


💰 Fetching SEC fundamentals...
----------------------------------------------------------------------
📊 Fetching fundamentals for 284 tickers...
🚀 Fetching SEC quarterly fundamental data...
📈 Adding fundamentals for 284 stocks...

📥 Fetching CIK mappings from SEC...
📥 Fetching SEC ticker-CIK mapping...
✅ Found 10142 ticker-CIK mappings
   ✅ Successfully fetched CIK mappings
   ✅ Found CIKs for 283/284 tickers
   ⚠️  Missing CIK for 1 tickers: ['IMB']
   💡 This is normal - some tickers may not have SEC filings

  [1/284] 📊 Fetching A...
      ✅ Completed: 0 | ❌ Failed: 0 | ⏱️  ETA: 0.0 min
      🔍 Fetching fundamentals for CIK: 0001090872
      ✅ A: Retrieved 47 quarterly records
      💾 Checkpoint saved: 1 tickers completed, 47 total records

  [2/284] 📊 Fetching AA...
      ✅ Completed: 1 | ❌ Failed: 0 | ⏱️  ETA: 370.3 min
      🔍 Fetching fundamentals for CIK: 0001675149
      ❌ AA: ZeroDivisionError: float division by zero
         Traceback: ZeroDivisionError: float division by ze

In [3]:
# 3. Fetch S&P 500 index data
print("\n" + "="*70)
print("📊 Fetching S&P 500 index data...")
print("-"*70)

sp500_index, sp500_metadata = fetch_sp500_index(
    start_date=DATA_START_DATE,
    end_date=DATA_END_DATE,
    verbose=True
)

if sp500_index is not None:
    print(f"\n✅ S&P 500 index fetched: {len(sp500_index):,} records")
    print(f"   📁 Saved to: {sp500_metadata['filepath']}")
    print(f"   📋 Latest copy: {sp500_metadata['latest_filepath']}")
else:
    print("\n❌ Failed to fetch S&P 500 index data")
    raise Exception("S&P 500 index fetching failed")



📊 Fetching S&P 500 index data...
----------------------------------------------------------------------
📊 Fetching S&P 500 index data from Yahoo Finance...
📅 Date range: 2012-01-01 to 2025-11-05
   📈 Ticker: ^GSPC
    ✅ Successfully fetched 3481 records
    📅 Date range: 2012-01-03 00:00:00 to 2025-11-04 00:00:00
    📈 Price range: $1277.06 - $6890.89

💾 Saving data with versioning...
   📦 Backed up existing file to: sp500_index_2012-2025_download_date_05_11_2025_v1.pkl
   📦 Backed up CSV to: sp500_index_2012-2025_download_date_05_11_2025_v2.csv
   ✅ Saved to: sp500_index_2012-2025_download_date_05_11_2025.pkl
   📋 Copied to latest: sp500_index_data_latest.pkl

✅ S&P 500 index data saved successfully!
   📄 Main file: sp500_index_2012-2025_download_date_05_11_2025.pkl
   📋 Latest copy: sp500_index_data_latest.pkl
   📈 Total return: 430.25%

✅ S&P 500 index fetched: 3,481 records
   📁 Saved to: data/research/sp500_index/sp500_index_2012-2025_download_date_05_11_2025.pkl
   📋 Latest copy

## 📊 Step 1 Summary: Data Fetching Results

In [3]:
print("="*70)
print("📊 STEP 1 SUMMARY: DATA FETCHING")
print("="*70)

print(f"\n📈 Stock Prices:")
print(f"   Total records: {len(stock_data):,}")
print(f"   Unique symbols: {stock_data['symbol'].nunique()}")
print(f"   Date range: {stock_data['date'].min()} to {stock_data['date'].max()}")
print(f"   Successful tickers: {len(stock_metadata['successful_tickers'])}")
print(f"   Failed tickers: {len(stock_metadata['failed_tickers'])}")
print(f"   File: {os.path.basename(stock_metadata['filepath'])}")

if sec_data is not None:
    print(f"\n💰 SEC Fundamentals:")
    print(f"   Total records: {len(sec_data):,}")
    if 'symbol' in sec_data.columns:
        print(f"   Unique symbols: {sec_data['symbol'].nunique()}")
    print(f"   File: {os.path.basename(sec_metadata['filepath'])}")
else:
    print(f"\n💰 SEC Fundamentals: Not available")

print(f"\n📊 S&P 500 Index:")
print(f"   Total records: {len(sp500_index):,}")
print(f"   Date range: {sp500_index['date'].min()} to {sp500_index['date'].max()}")
print(f"   Total return: {sp500_metadata['data_info']['total_return_pct']:.2f}%")
print(f"   File: {os.path.basename(sp500_metadata['filepath'])}")

print(f"\n✅ Data fetching complete! Ready for training.")


📊 STEP 1 SUMMARY: DATA FETCHING

📈 Stock Prices:
   Total records: 905,852
   Unique symbols: 284
   Date range: 2012-01-03 00:00:00 to 2025-11-03 00:00:00
   Successful tickers: 284
   Failed tickers: 0
   File: sp500_stock_data_latest.pkl


NameError: name 'sec_data' is not defined

## 🎓 Step 2: Train XGBoost Models (2012-2021)

In [ ]:
print("="*70)
print("🎓 STEP 2: MODEL TRAINING")
print("="*70)
print(f"🤖 Model type: {MODEL_TYPES}")
print(f"📊 Training split: {TRAIN_SPLIT*100:.0f}% train, {(1-TRAIN_SPLIT)*100:.0f}% validation")
print()

# Train only XGBoost models
train_results = train_all_models_optimized(
    df=stock_data,  # Pass the fetched stock data
    feature_windows=FEATURE_WINDOWS,
    target_windows=TARGET_WINDOWS,
    model_types=MODEL_TYPES,
    train_split=TRAIN_SPLIT,
    model_suffix=MODEL_SUFFIX,
    force_recalculate_features=True,  # Use cached features if available
    verbose=True
)

print("\n✅ Training complete!")


🎓 STEP 2: MODEL TRAINING
🤖 Model type: ['XGBoost']
📊 Training split: 80% train, 20% validation


🚀 STARTING OPTIMIZED TRAINING PIPELINE
📊 Total models to train: 36
   Feature windows: [5, 10, 15, 20, 25, 30]
   Target windows: [5, 10, 15, 20, 25, 30]
   Model types: ['XGBoost']
   Model suffix: 2012_2021
   Train split: 80% train, 20% test

💾 Feature cache directory: data/research/features_cache

STEP 1: PRE-CALCULATING FEATURES (with caching)

🔧 PRE-CALCULATING FEATURES FOR ALL WINDOWS
⚠️  TIMING FIX: Features use prices shifted by 1 trading day (T uses data ≤ T-1)
⚠️  Force recalculate enabled - ignoring cache

📊 Window 5 days:
   🔧 Calculating features for 5-day window...

  📊 Creating comprehensive features for 5-day window...
    🔧 Calculating momentum features...
    📈 Calculating momentum features for 5-day window...
       ⚠️  Using prices shifted by 1 trading day (T uses data ≤ T-1)
    ✅ Generated 14 momentum features
    🔧 Calculating reversal features...
    📉 Calculating r

## 📊 Step 2 Summary: Training Results

In [ ]:
print("="*70)
print("📊 STEP 2 SUMMARY: MODEL TRAINING")
print("="*70)

# Load training log to see results
import json
from training_util.cache_manager import load_training_log

training_log = load_training_log()

if training_log:
    # Filter for models with configured suffix
    xgboost_models = [m for m in training_log 
                     if any(model_type in m.get('model_key', '') for model_type in MODEL_TYPES)
                     and MODEL_SUFFIX in m.get('model_key', '')]
    
    print(f"\n📊 Total XGBoost models trained: {len(xgboost_models)}")
    
    if xgboost_models:
        # Sort by accuracy (if available)
        if 'test_accuracy' in xgboost_models[0]:
            xgboost_models.sort(key=lambda x: x.get('test_accuracy', 0), reverse=True)
        
        print(f"\n🏆 Top 10 Models by Accuracy:")
        print("-"*70)
        print(f"{'Model Key':<50} {'Accuracy':<12} {'F1 Score':<12}")
        print("-"*70)
        
        for model in xgboost_models[:10]:
            model_key = model.get('model_key', 'N/A')
            accuracy = model.get('test_accuracy', 0) * 100 if 'test_accuracy' in model else 0
            f1_score = model.get('test_f1_score', 0) if 'test_f1_score' in model else 0
            print(f"{model_key:<50} {accuracy:>10.2f}% {f1_score:>11.4f}")
        
        # Summary statistics
        if 'test_accuracy' in xgboost_models[0]:
            accuracies = [m.get('test_accuracy', 0) * 100 for m in xgboost_models]
            print(f"\n📈 Accuracy Statistics:")
            print(f"   Mean: {np.mean(accuracies):.2f}%")
            print(f"   Median: {np.median(accuracies):.2f}%")
            print(f"   Min: {np.min(accuracies):.2f}%")
            print(f"   Max: {np.max(accuracies):.2f}%")
    else:
        print("\n⚠️  No XGBoost models found in training log")
else:
    print("\n⚠️  Training log not found or empty")

print(f"\n✅ Training summary complete! Ready for backtesting.")


## 🧪 Step 3: Backtesting (2022-Today) with 15% Stop-Loss

In [ ]:
print("="*70)
print("🧪 STEP 3: BACKTESTING")
print("="*70)
print(f"📅 Test period: {BACKTEST_START_DATE} to {BACKTEST_END_DATE}")
print(f"🛡️  Stop-loss: {STOP_LOSS*100:.0f}%")
print(f"📊 Models: {MODEL_TYPES} models trained on {DATA_START_DATE} to 2021-12-31")
print()

# Run backtesting with stop-loss enabled
backtest_results = run_ultra_optimized_backtesting(
    test_start_date=BACKTEST_START_DATE,
    test_end_date=BACKTEST_END_DATE,
    max_positions=MAX_POSITIONS,
    verbose=True,
    resume=True,  # Skip already-completed models
    trade_when_positions_zero=TRADE_WHEN_POSITIONS_ZERO,
    include_randomforest=False,  # Only configured model types
    include_xgboost=('XGBoost' in MODEL_TYPES),
    model_suffix=MODEL_SUFFIX,
    stop_loss=STOP_LOSS,
    results_dir=RESULTS_DIR
)

print("\n✅ Backtesting complete!")


## 📊 Step 3 Summary: Backtesting Results

In [ ]:
print("="*70)
print("📊 STEP 3 SUMMARY: BACKTESTING RESULTS")
print("="*70)

if backtest_results is not None and len(backtest_results) > 0:
    print(f"\n📊 Total models backtested: {len(backtest_results)}")
    
    # Sort by total return
    backtest_results_sorted = backtest_results.sort_values('total_return', ascending=False)
    
    print(f"\n🏆 Top 10 Performing Models:")
    print("-"*70)
    print(f"{'Model Key':<50} {'Total Return':<15} {'Trades':<10}")
    print("-"*70)
    
    for idx, row in backtest_results_sorted.head(10).iterrows():
        model_key = row.get('model_key', 'N/A')
        total_return = row.get('total_return', 0) * 100
        total_trades = row.get('total_trades', 0)
        print(f"{model_key:<50} {total_return:>13.2f}% {total_trades:>9.0f}")
    
    # Summary statistics
    returns = backtest_results_sorted['total_return'].values * 100
    print(f"\n📈 Return Statistics:")
    print(f"   Mean: {np.mean(returns):.2f}%")
    print(f"   Median: {np.median(returns):.2f}%")
    print(f"   Min: {np.min(returns):.2f}%")
    print(f"   Max: {np.max(returns):.2f}%")
    
    # Count profitable models
    profitable = (returns > 0).sum()
    print(f"\n💰 Profitable Models: {profitable}/{len(returns)} ({profitable/len(returns)*100:.1f}%)")
    
    # Stop-loss statistics
    if 'stop_loss_triggered' in backtest_results_sorted.columns:
        total_stop_losses = backtest_results_sorted['stop_loss_triggered'].sum()
        total_trades_all = backtest_results_sorted['total_trades'].sum()
        if total_trades_all > 0:
            stop_loss_rate = (total_stop_losses / total_trades_all) * 100
            print(f"\n🛡️  Stop-Loss Statistics:")
            print(f"   Total stop-losses triggered: {total_stop_losses:,.0f}")
            print(f"   Stop-loss rate: {stop_loss_rate:.2f}%")
    
    print(f"\n✅ Backtesting summary complete!")
else:
    print("\n⚠️  No backtesting results found")

print(f"\n🎉 Full pipeline complete!")


## 🎉 Pipeline Complete!

### Summary of All Steps:

1. ✅ **Data Fetching**: Downloaded stock prices, SEC fundamentals, and S&P 500 index (2012-today)
2. ✅ **Model Training**: Trained XGBoost models on 2012-2021 data with 70/30 split
3. ✅ **Backtesting**: Tested models on 2022-today period with 15% stop-loss

### Next Steps:
- Review top-performing models from backtesting results
- Analyze detailed trade logs for best models
- Compare portfolio performance vs S&P 500 index
- Fine-tune models or test different stop-loss levels